# Gesture Classifier Training → ONNX Export (Unity-ready)
FYP-26-S2-14 — Gaming with Bare Hands (TaoSL)

This notebook trains **two separate YOLO11s classification models** (left-hand movement, right-hand Mudra skill) starting from the ImageNet-pretrained `yolo11s-cls.pt` checkpoint, then exports both to **ONNX (opset 15, 224x224 fixed input)** for Unity Sentis / Inference Engine.

**Before running:**
1. Upload your dataset to Kaggle as a Dataset, with this structure:
```
<dataset-root>/
  left/
    train/<class_name>/*.jpg
    val/<class_name>/*.jpg
  right/
    train/<class_name>/*.jpg
    val/<class_name>/*.jpg
```
2. Add the dataset to this notebook (Add Data, top right panel).
3. Turn on GPU: **Settings (right panel) → Accelerator → GPU T4 x2** (or whichever GPU is available).
4. Edit the path variables in the **Config** cell below to match your dataset's actual folder name.


In [ ]:
# --- Install dependencies ---
!pip install -q ultralytics onnx onnxruntime onnxslim


In [ ]:
# --- Sanity check: confirm GPU is available ---
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - go to Settings > Accelerator and enable a GPU")


## Config
Edit `DATASET_ROOT` to match the dataset name Kaggle assigns after you add it (visible under `/kaggle/input/`).


In [ ]:
from pathlib import Path

# EDIT THIS to match your uploaded dataset folder name under /kaggle/input/
DATASET_ROOT = Path("/kaggle/input/gesture-dataset")

LEFT_DATA_DIR = DATASET_ROOT / "left"
RIGHT_DATA_DIR = DATASET_ROOT / "right"

# Base pretrained checkpoint (ImageNet weights) - starting point for transfer learning
BASE_CHECKPOINT = "yolo11s-cls.pt"

IMG_SIZE = 224
EPOCHS = 100          # ultralytics has early-stopping (patience) built in, so this is a ceiling not a guarantee
PATIENCE = 20         # stop early if val accuracy hasn't improved in this many epochs
BATCH = 64
ONNX_OPSET = 15       # safe/stable range for Unity Sentis (documented support: opset 7-15)

OUTPUT_DIR = Path("/kaggle/working/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Left data dir exists:", LEFT_DATA_DIR.exists())
print("Right data dir exists:", RIGHT_DATA_DIR.exists())


## Train Left-Hand Model (movement/dash classifier)
Classes expected: left_stop, left_front, left_behind, left_left, left_right, and the four "dash" variants (per your PRD's 9 left classes).


In [ ]:
from ultralytics import YOLO

left_model = YOLO(BASE_CHECKPOINT)

left_results = left_model.train(
    data=str(LEFT_DATA_DIR),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=str(OUTPUT_DIR),
    name="left_train",
    exist_ok=True,
)

print("Left model training complete.")
print("Best weights:", left_model.trainer.best)


## Train Right-Hand Model (Mudra skill classifier)
Classes expected: fist/stop, iron, young, flow, burst, ground, confirm, cancel (per your PRD's right-hand classes).


In [ ]:
right_model = YOLO(BASE_CHECKPOINT)

right_results = right_model.train(
    data=str(RIGHT_DATA_DIR),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=str(OUTPUT_DIR),
    name="right_train",
    exist_ok=True,
)

print("Right model training complete.")
print("Best weights:", right_model.trainer.best)


## Validate Both Models
Quick check of top-1 / top-5 accuracy on the val split before exporting — catch problems here rather than after ONNX conversion.


In [ ]:
left_best = YOLO(left_model.trainer.best)
right_best = YOLO(right_model.trainer.best)

print("=== Left model val metrics ===")
left_metrics = left_best.val(data=str(LEFT_DATA_DIR), imgsz=IMG_SIZE)
print("top1:", left_metrics.top1, " top5:", left_metrics.top5)

print("\n=== Right model val metrics ===")
right_metrics = right_best.val(data=str(RIGHT_DATA_DIR), imgsz=IMG_SIZE)
print("top1:", right_metrics.top1, " top5:", right_metrics.top5)


## Export Both Models to ONNX
Fixed input size (no dynamic shape), opset 15, for Unity Sentis / Inference Engine compatibility.


In [ ]:
left_onnx_path = left_best.export(format="onnx", imgsz=IMG_SIZE, opset=ONNX_OPSET, dynamic=False, simplify=True)
right_onnx_path = right_best.export(format="onnx", imgsz=IMG_SIZE, opset=ONNX_OPSET, dynamic=False, simplify=True)

print("Left ONNX exported to:", left_onnx_path)
print("Right ONNX exported to:", right_onnx_path)


## Save Class Name Mappings
Unity needs to know which output index maps to which gesture name (e.g. index 0 = "iron"). ONNX doesn't carry this info on its own, so we save it alongside the models as JSON — hand this to your Unity teammate together with the .onnx files.


In [ ]:
import json
import shutil

left_names = left_best.names   # dict: {0: 'iron', 1: 'young', ...}
right_names = right_best.names

with open(OUTPUT_DIR / "left_class_names.json", "w") as f:
    json.dump(left_names, f, indent=2)

with open(OUTPUT_DIR / "right_class_names.json", "w") as f:
    json.dump(right_names, f, indent=2)

# Copy the ONNX files into a single flat folder for easy download
shutil.copy(left_onnx_path, OUTPUT_DIR / "left_gesture_model.onnx")
shutil.copy(right_onnx_path, OUTPUT_DIR / "right_gesture_model.onnx")

print("Left classes:", left_names)
print("Right classes:", right_names)
print("\nFiles ready in:", OUTPUT_DIR)
!ls -la {OUTPUT_DIR}


## Sanity Check: Run ONNX Model on a Sample Image
Confirms the exported .onnx file actually loads and produces sensible predictions with `onnxruntime`, matching what the .pt model would predict — this is the same runtime approach you'd use to test locally on your laptop later (CPU-only `onnxruntime` works fine, no GPU needed).


In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image
import random

def preprocess(image_path, size=IMG_SIZE):
    img = Image.open(image_path).convert("RGB").resize((size, size))
    arr = np.array(img).astype(np.float32) / 255.0
    arr = arr.transpose(2, 0, 1)  # HWC -> CHW
    arr = np.expand_dims(arr, axis=0)  # add batch dim
    return arr

def sanity_check(onnx_path, val_dir, class_names, label=""):
    session = ort.InferenceSession(str(onnx_path))
    input_name = session.get_inputs()[0].name

    # pick a random image from the val set to test
    val_path = Path(val_dir) / "val"
    class_folders = [d for d in val_path.iterdir() if d.is_dir()]
    sample_class = random.choice(class_folders)
    sample_images = list(sample_class.glob("*.jpg")) + list(sample_class.glob("*.png"))
    sample_img = random.choice(sample_images)

    x = preprocess(sample_img)
    outputs = session.run(None, {input_name: x})
    probs = outputs[0][0]
    top_idx = int(np.argmax(probs))

    print(f"[{label}] Test image: {sample_img.name} (true folder: {sample_class.name})")
    print(f"[{label}] ONNX predicted: {class_names[top_idx]}  (confidence: {probs[top_idx]:.3f})")
    print()

sanity_check(OUTPUT_DIR / "left_gesture_model.onnx", LEFT_DATA_DIR, left_names, label="LEFT")
sanity_check(OUTPUT_DIR / "right_gesture_model.onnx", RIGHT_DATA_DIR, right_names, label="RIGHT")


## Download Your Models
Zip everything up so you can download it in one go from the Kaggle notebook's Output panel.


In [ ]:
import shutil

shutil.make_archive("/kaggle/working/gesture_models_export", "zip", OUTPUT_DIR)
print("Zip ready: gesture_models_export.zip")
print("Download it from the Output panel on the right side of this Kaggle notebook.")


## Next Steps (on your laptop)
1. Download `gesture_models_export.zip` from Kaggle's Output panel.
2. Unzip — you'll have `left_gesture_model.onnx`, `right_gesture_model.onnx`, and their `*_class_names.json` files.
3. Test locally with `onnxruntime` (CPU) before handing off to Unity:
   ```
   pip install onnxruntime
   python -c "import onnxruntime as ort; s = ort.InferenceSession('left_gesture_model.onnx'); print(s.get_inputs()[0].shape)"
   ```
4. Hand the `.onnx` files + `.json` class mappings to your Unity teammate for Sentis/Inference Engine import.
